# 02 — Model architecture, logits, and loss

**Goal:** understand how a small CNN turns an image batch into raw class scores, and how the loss function tells the model whether it is wrong.

This notebook covers:

1. A visual CNN architecture diagram
2. Tensor-shape flow through the layers
3. Feature maps from a convolution layer
4. Logits and softmax probabilities
5. Cross-entropy loss
6. How this connects to the training loop


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
import numpy as np
import pandas as pd

plt.rcParams["figure.figsize"] = (7, 4)


## 1. Define a small CNN for inspection

This demo model matches the model used by `train.py`:

```text
image batch → convolution blocks → flatten → linear classifier → logits
```

The final output is **not** an image. It is one score per class.


In [ ]:
class SmallCNNDemo(nn.Module):
    def __init__(self, num_classes=4, image_size=32, base_channels=16):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(base_channels, base_channels * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        feature_size = image_size // 4
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_channels * 2 * feature_size * feature_size, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        logits = self.classifier(x)
        return logits

model = SmallCNNDemo(num_classes=4, image_size=32, base_channels=16)
print(model)


## 2. Visualize the CNN layer flow

This diagram is a simplified teaching diagram. It is easier to read than a full computation graph and shows the main idea: convolution blocks gradually convert the image into learned feature representations, then the classifier produces one score per class.


In [ ]:
def draw_cnn_architecture():
    layers = [
        ("Input image batch", "[B, 3, 32, 32]", "RGB images"),
        ("Conv + ReLU", "[B, 16, 32, 32]", "local filters"),
        ("MaxPool", "[B, 16, 16, 16]", "downsample"),
        ("Conv + ReLU", "[B, 32, 16, 16]", "more features"),
        ("MaxPool", "[B, 32, 8, 8]", "downsample"),
        ("Flatten", "[B, 2048]", "vector"),
        ("Linear", "[B, 4]", "logits"),
    ]

    fig, ax = plt.subplots(figsize=(13, 3.4))
    ax.set_xlim(0, len(layers) * 2.1)
    ax.set_ylim(0, 3)
    ax.axis("off")

    x = 0.2
    for i, (name, shape, note) in enumerate(layers):
        width = 1.55
        rect = Rectangle((x, 1.0), width, 1.0, fill=False, linewidth=1.8)
        ax.add_patch(rect)
        ax.text(x + width / 2, 1.70, name, ha="center", va="center", fontsize=9, weight="bold")
        ax.text(x + width / 2, 1.38, shape, ha="center", va="center", fontsize=8)
        ax.text(x + width / 2, 1.12, note, ha="center", va="center", fontsize=7)
        if i < len(layers) - 1:
            arrow = FancyArrowPatch((x + width + 0.08, 1.5), (x + width + 0.43, 1.5), arrowstyle="->", mutation_scale=12, linewidth=1.4)
            ax.add_patch(arrow)
        x += 2.05

    ax.set_title("Small CNN architecture: image batch → logits", fontsize=13, pad=14)
    plt.show()

draw_cnn_architecture()


### What the blocks mean

- **Convolution** uses learnable filters to detect local patterns.
- **ReLU** adds non-linearity, allowing the network to learn more than a linear transformation.
- **MaxPool** reduces image resolution and keeps strong local responses.
- **Flatten** converts feature maps into a vector.
- **Linear classifier** converts the vector into one raw score per class.


## 3. Check actual layer-by-layer tensor shapes

The diagram is conceptual. The next cell passes a fake batch through the model and records the actual input/output shape at each layer.


In [ ]:
def collect_layer_shapes(model, example_batch):
    records = []
    hooks = []

    def make_hook(name):
        def hook(module, inputs, output):
            input_shape = tuple(inputs[0].shape)
            output_shape = tuple(output.shape)
            records.append({
                "layer": name,
                "module": module.__class__.__name__,
                "input_shape": input_shape,
                "output_shape": output_shape,
            })
        return hook

    for name, module in model.named_modules():
        if name and len(list(module.children())) == 0:
            hooks.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        _ = model(example_batch)

    for hook in hooks:
        hook.remove()

    return pd.DataFrame(records)

fake_batch = torch.randn(64, 3, 32, 32)
shape_table = collect_layer_shapes(model, fake_batch)
display(shape_table)


In [ ]:
# Optional Keras-style summary.
# If this fails, the shape table above is enough.
try:
    from torchinfo import summary
    summary(model, input_size=(64, 3, 32, 32), col_names=["input_size", "output_size", "num_params"])
except Exception as exc:
    print("torchinfo summary unavailable:", exc)


## 4. Load one CIFAR batch

Now use real CIFAR images. If this fails, run this first from the project root:

```bash
python prepare_cifar10_subset.py --train-per-class 400 --val-per-class 100
```


In [ ]:
DATA_DIR = PROJECT_ROOT / "data_cifar"
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(DATA_DIR / "train", transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
images, labels = next(iter(loader))
class_names = dataset.classes

print("Image batch:", images.shape)
print("Labels:", labels)
print("Class names:", class_names)


## 5. Visualize feature maps from the first convolution

A convolution layer does not output a single image. It outputs multiple **feature maps**.

If the first convolution has 16 output channels, it produces 16 feature maps per image. Each feature map responds to a different learned filter.


In [ ]:
first_image = images[0:1]
first_conv = model.features[0]

with torch.no_grad():
    feature_maps = first_conv(first_image)

print("Original image shape:", tuple(first_image.shape))
print("Feature maps shape:", tuple(feature_maps.shape))

fig, axes = plt.subplots(1, 5, figsize=(12, 2.5))
axes[0].imshow(first_image[0].permute(1, 2, 0))
axes[0].set_title("Input image")
axes[0].axis("off")

for i in range(4):
    axes[i + 1].imshow(feature_maps[0, i].detach().numpy(), cmap="gray")
    axes[i + 1].set_title(f"Feature map {i}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()


## 6. What are logits?

The model output is called **logits**.

For one image, the CNN returns one number per class:

```text
[score_airplane, score_automobile, score_cat, score_ship]
```

Important properties of logits:

```text
- logits are raw class scores
- logits are not probabilities
- logits can be negative
- logits do not need to sum to 1
- the largest logit gives the predicted class
```

To interpret the scores as probabilities, we apply **softmax**.


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(images)
    probabilities = torch.softmax(logits, dim=1)
    predicted_indices = probabilities.argmax(dim=1)

print("Input image batch shape:", tuple(images.shape))
print("Logits shape:", tuple(logits.shape))
print("Softmax probabilities shape:", tuple(probabilities.shape))
print("\nLogits for first image:")
print(logits[0])
print("\nSoftmax probabilities for first image:")
print(probabilities[0])
print("\nProbability sum:", probabilities[0].sum().item())
print("Predicted class:", class_names[int(predicted_indices[0])])
print("True class:", class_names[int(labels[0])])

## 7. Plot logits and softmax probabilities

The two plots below show the same model output in two forms.

- **Left:** raw logits. These are the values used by the loss function.
- **Right:** softmax probabilities. These are easier for humans to interpret.

At this point the model is still randomly initialized, so the prediction may be wrong.


In [ ]:
example_index = 0
logit_values = logits[example_index].detach().cpu().numpy()
prob_values = probabilities[example_index].detach().cpu().numpy()
true_idx = int(labels[example_index])
pred_idx = int(predicted_indices[example_index])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].imshow(images[example_index].permute(1, 2, 0))
axes[0].set_title(f"Image\ntrue: {class_names[true_idx]}")
axes[0].axis("off")

axes[1].bar(class_names, logit_values)
axes[1].set_title("Raw logits")
axes[1].set_ylabel("Score")
axes[1].tick_params(axis="x", rotation=30)

axes[2].bar(class_names, prob_values)
axes[2].set_title("Softmax probabilities")
axes[2].set_ylabel("Probability")
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis="x", rotation=30)

plt.suptitle(f"Predicted class: {class_names[pred_idx]}")
plt.tight_layout()
plt.show()

## 8. What is a loss function?

A loss function turns the model output into one number that tells us how wrong the model is.

For one image in a classification problem, the idea is:

```text
image → CNN → logits → softmax probabilities → probability of the true class → loss
```

For cross-entropy loss, the loss for one image is small when the model assigns high probability to the true class, and large when it assigns low probability to the true class.

The important PyTorch rule is:

```python
loss = nn.CrossEntropyLoss()(logits, labels)
```

`CrossEntropyLoss` expects raw logits. We use softmax here only for visualization.

In [ ]:
criterion = nn.CrossEntropyLoss()

# Use one image from the current batch.
one_image = images[0:1]
one_label = labels[0:1]
true_class_index = int(one_label.item())
true_class_name = class_names[true_class_index]

with torch.no_grad():
    one_logits = model(one_image)
    one_probs = torch.softmax(one_logits, dim=1)
    one_loss = criterion(one_logits, one_label)

logit_values = one_logits.squeeze(0).detach().cpu().numpy()
prob_values = one_probs.squeeze(0).detach().cpu().numpy()
model_probability_true_class = float(prob_values[true_class_index])
loss_value = float(one_loss.item())
predicted_index = int(prob_values.argmax())
predicted_class_name = class_names[predicted_index]

print(f"True class: {true_class_name}")
print(f"Predicted class: {predicted_class_name}")
print(f"Model probability for the true class: {model_probability_true_class:.4f}")
print(f"Ideal target probability for the true class: 1.0000")
print(f"Cross-entropy loss: {loss_value:.4f}")


### One-image view of cross-entropy loss

The plot below shows how the loss is produced for a single image.

1. The CNN outputs one **logit** for each class.
2. Softmax converts the logits into probabilities.
3. The **true class** has a one-hot target, so its ideal target probability is **1**.
4. The model usually gives a smaller probability to that true class, call it $p_t$.
5. Cross-entropy turns that one number into one scalar loss.

Mathematically, for one image:

```text
loss = -log(probability assigned to the true class) = -log(p_t)
```

So a higher model probability for the true class gives a smaller loss.

> Note: the orange target box helps you compare the **ideal target probability = 1** with the **model probability for the true class = $p_t$**. The loss is **not** a simple vertical difference between them; it is computed as $-\log(p_t)$.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
x = np.arange(len(class_names))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

axes[0].bar(x, logit_values, alpha=0.85)
axes[0].axhline(0.0, linewidth=1)
axes[0].set_xticks(x)
axes[0].set_xticklabels(class_names, rotation=25, ha="right")
axes[0].set_ylabel("Raw score")
axes[0].set_title("Step 1: CNN outputs logits")

bar_width = 0.8
axes[1].bar(x, prob_values, alpha=0.85, label="Softmax probability")
target_left = true_class_index - bar_width / 2
target_box = Rectangle(
    (target_left, 0),
    bar_width,
    1.0,
    fill=False,
    edgecolor="tab:orange",
    linewidth=2.0,
    linestyle=":",
)
axes[1].add_patch(target_box)
axes[1].scatter(
    [true_class_index],
    [model_probability_true_class],
    s=180,
    marker="*",
    color="tab:orange",
    label=f"Model probability for true class = {model_probability_true_class:.3f}",
    zorder=3,
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(class_names, rotation=25, ha="right")
axes[1].set_ylabel("Probability")
axes[1].set_ylim(0, 1.05)
axes[1].set_title(f"Step 2: true class = {true_class_name}\nloss = -log({model_probability_true_class:.3f}) = {loss_value:.3f}")
from matplotlib.lines import Line2D
legend_handles = [
    Rectangle((0, 0), 1, 1, facecolor=plt.rcParams['axes.prop_cycle'].by_key()['color'][0], alpha=0.85, label='Softmax probability'),
    Rectangle((0, 0), 1, 1, fill=False, edgecolor='tab:orange', linewidth=2.0, linestyle=':', label='Ideal target for true class = 1'),
    Line2D([0], [0], marker='*', color='tab:orange', linestyle='None', markersize=12, label=f'Model probability for true class = {model_probability_true_class:.3f}'),
]
axes[1].legend(handles=legend_handles, loc="upper left", fontsize=8)

fig.suptitle("From logits and label to one cross-entropy loss", y=1.05)
plt.tight_layout()
plt.show()


### Cross-entropy expects logits, not softmax probabilities

In PyTorch, this is correct:

```python
logits = model(images)
loss = nn.CrossEntropyLoss()(logits, labels)
```

Do **not** manually apply softmax before `CrossEntropyLoss`. PyTorch performs the stable log-softmax calculation internally.

The probability plot is only for understanding and interpretation. The training code still passes **raw logits** directly to the loss function.


## 9. Prediction grid from the untrained model

The model has not been trained yet, so the predictions may look random. That is expected.

Training will update the convolution filters and classifier weights so that the correct class usually receives a larger logit and lower loss.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(9, 4.8))
axes = axes.reshape(-1)

for i, ax in enumerate(axes):
    ax.imshow(images[i].permute(1, 2, 0))
    true_name = class_names[int(labels[i])]
    pred_name = class_names[int(predicted_indices[i])]
    conf = float(probabilities[i].max())
    ax.set_title(f"true: {true_name}\npred: {pred_name}\nconf: {conf:.2f}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 10. How this connects to `train.py`

The core training step in `train.py` is:

```python
logits = model(images)
loss = criterion(logits, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()
```

Meaning:

```text
model(images)        → produce logits
criterion(...)       → measure how wrong the model is
loss.backward()      → compute gradients
optimizer.step()     → update model parameters
```

## Student checkpoint

Before training, answer:

1. What is the model input shape?
2. What is the model output shape?
3. Why are logits not probabilities?
4. What does softmax do?
5. What does cross-entropy loss penalize?
6. Why should `CrossEntropyLoss` receive logits directly?
